# linear-affine-on-custom-tensor — worked example 2: Weight backward for the affine map

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `linear-affine-on-custom-tensor`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

For `out = x @ weight`, the gradient of the loss w.r.t. `weight` is `x.T @ grad_out`. The batch axis is contracted by that matmul, so a `(B, out)` upstream gradient and a `(B, in)` input give a `(in, out)` weight gradient — exactly the weight's shape.

## Worked solution

We build the backward primitive that pairs with the forward matmul.

1. **Shapes.** `x` is `(B, in)`, `grad_out` is `(B, out)`. We want `grad_weight` shaped `(in, out)`.
2. **The rule.** Differentiating `out = x @ weight` gives `dL/dweight = x.T @ grad_out`. Transposing `x` to `(in, B)` and matmul-ing with `(B, out)` contracts the batch and lands on `(in, out)`.
3. **Shape check.** We assert the result equals `weight.array.shape` so a wrong transpose is caught immediately.
4. **Verification.** We cross-check against a finite-difference estimate of `d(sum(x@weight))/dweight`. For the sum-reduced loss, `grad_out` is all ones and the analytic weight grad equals `x.sum(axis=0)` broadcast across columns; the numerical gradient confirms it.

The print shows the analytic and numerical gradients agree to tolerance.

In [ ]:
import numpy as np
from dataclasses import dataclass

np.random.seed(1)

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad
        self.recipe = recipe

def linear_weight_backward(grad_out, x, weight):
    gw = x.array.T @ grad_out
    assert gw.shape == weight.array.shape, (gw.shape, weight.array.shape)
    return gw

x = MiniTensor(np.random.randn(5, 3))
weight = MiniTensor(np.random.randn(3, 4), requires_grad=True)
grad_out = np.ones((5, 4))
gw = linear_weight_backward(grad_out, x, weight)

# numerical check on loss = sum(x @ weight)
eps = 1e-6
num = np.zeros_like(weight.array)
for i in range(3):
    for j in range(4):
        wp = weight.array.copy(); wp[i, j] += eps
        wm = weight.array.copy(); wm[i, j] -= eps
        num[i, j] = ((x.array @ wp).sum() - (x.array @ wm).sum()) / (2 * eps)
print('grad shape:', gw.shape, '| matches numeric:', np.allclose(gw, num, atol=1e-4))